In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Loading data set 
The main target variable is Burn Rate, which represents the employee burnout level as a continuous outcome

In [3]:
df = pd.read_csv("train.csv")

print(f"Dataset shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns")
df.head()

Dataset shape: 22,750 rows and 9 columns


,Employee ID,Date of Joining,Gender,Company Type,WFH Setup Available,Designation,Resource Allocation,Mental Fatigue Score,Burn Rate
0,fffe32003000360033003200,2008-09-30,Female,Service,No,2.000,3.000,3.800,0.160
1,fffe3700360033003500,2008-11-30,Male,Service,Yes,1.000,2.000,5.000,0.360
2,fffe31003300320037003900,2008-03-10,Female,Product,Yes,2.000,NaN,5.800,0.490
3,fffe32003400380032003900,2008-11-03,Male,Service,Yes,1.000,1.000,2.600,0.200
4,fffe31003900340031003600,2008-07-24,Female,Service,No,3.000,7.000,6.900,0.520


## Understanding Data Structure 


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22750 entries, 0 to 22749
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Employee ID           22750 non-null  str    
 1   Date of Joining       22750 non-null  str    
 2   Gender                22750 non-null  str    
 3   Company Type          22750 non-null  str    
 4   WFH Setup Available   22750 non-null  str    
 5   Designation           22750 non-null  float64
 6   Resource Allocation   21369 non-null  float64
 7   Mental Fatigue Score  20633 non-null  float64
 8   Burn Rate             21626 non-null  float64
dtypes: float64(4), str(5)
memory usage: 1.6 MB


In [5]:
missing_summary = (
    df.isnull()
    .sum()
    .reset_index()
    .rename(columns={"index": "Column", 0: "Missing Values"})
)

missing_summary["Missing %"] = (missing_summary["Missing Values"] / len(df) * 100).round(2)
missing_summary = missing_summary.sort_values("Missing %", ascending=False)

missing_summary

,Column,Missing Values,Missing %
7,Mental Fatigue Score,2117,9.310
6,Resource Allocation,1381,6.070
8,Burn Rate,1124,4.940
0,Employee ID,0,0.000
1,Date of Joining,0,0.000
4,WFH Setup Available,0,0.000
3,Company Type,0,0.000
2,Gender,0,0.000
5,Designation,0,0.000


In [6]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


## Data Cleaning and Feature Preparation
The cleaning choices are intentionally transparent so they can later be aligned with the modeling pipeline.

In [7]:
eda_df = df.copy()

eda_df.columns = (
    eda_df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

In [8]:
# Converting joining date
if "Date_of_Joining" in eda_df.columns:
    eda_df["Date_of_Joining"] = pd.to_datetime(eda_df["Date_of_Joining"], errors="coerce")
    reference_date = eda_df["Date_of_Joining"].max()
    eda_df["Tenure_Days"] = (reference_date - eda_df["Date_of_Joining"]).dt.days
    eda_df["Join_Month_Year"] = eda_df["Date_of_Joining"].dt.to_period("M").astype(str)


eda_df = eda_df.dropna(subset=["Burn_Rate"])

# burnout categories 
eda_df["Burnout_Risk_Level"] = pd.cut(
    eda_df["Burn_Rate"],
    bins=[-np.inf, 0.31, 0.59, np.inf],
    labels=["Low Risk", "Moderate Risk", "High Risk"]
)

# High burnout indicator based on median burn rate
median_burn = eda_df["Burn_Rate"].median()
eda_df["High_Burnout"] = eda_df["Burn_Rate"] > median_burn

eda_df.head()

,Employee_ID,Date_of_Joining,Gender,Company_Type,WFH_Setup_Available,Designation,Resource_Allocation,Mental_Fatigue_Score,Burn_Rate,Tenure_Days,Join_Month_Year,Burnout_Risk_Level,High_Burnout
0,fffe32003000360033003200,2008-09-30,Female,Service,No,2.000,3.000,3.800,0.160,92,2008-09,Low Risk,False
1,fffe3700360033003500,2008-11-30,Male,Service,Yes,1.000,2.000,5.000,0.360,31,2008-11,Moderate Risk,False
2,fffe31003300320037003900,2008-03-10,Female,Product,Yes,2.000,NaN,5.800,0.490,296,2008-03,Moderate Risk,True
3,fffe32003400380032003900,2008-11-03,Male,Service,Yes,1.000,1.000,2.600,0.200,58,2008-11,Low Risk,False
4,fffe31003900340031003600,2008-07-24,Female,Service,No,3.000,7.000,6.900,0.520,160,2008-07,Moderate Risk,True


In [9]:
numeric_cols = eda_df.select_dtypes(include=np.number).columns.tolist()
eda_df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Designation,21626.000,2.179,1.135,0.000,1.000,2.000,3.000,5.000
Resource_Allocation,20348.000,4.484,2.048,1.000,3.000,4.000,6.000,10.000
Mental_Fatigue_Score,19681.000,5.730,1.921,0.000,4.600,5.900,7.100,10.000
Burn_Rate,21626.000,0.452,0.198,0.000,0.310,0.450,0.590,1.000
Tenure_Days,21626.000,182.638,105.383,0.000,92.000,182.000,275.000,365.000
